In [0]:
CREATE OR REPLACE VIEW 03_dev_gold.kpi_models.kpi_overall_view AS
SELECT
    ROUND(COUNT(DISTINCT order_id),2) as total_order_count,
    ROUND(SUM(total_amount),2) AS total_revenue,
    COUNT(DISTINCT CASE WHEN order_status = 'COMPLETED' THEN order_id END) AS completed_order_count,
    ROUND(COUNT(DISTINCT CASE WHEN order_status = 'COMPLETED' THEN order_id END)/COUNT(DISTINCT order_id),2) AS completed_order_rate,
    ROUND(SUM(total_amount)  / COUNT(DISTINCT order_id),2) AS average_order_value,
    COUNT(DISTINCT total_amount) AS active_customers
FROM 03_dev_gold.kpi_models.data_cube;


CREATE OR REPLACE VIEW 03_dev_gold.kpi_models.kpi_revenue_by_country AS
SELECT
    order_country AS country,
    ROUND(SUM(total_amount),2) AS revenue
FROM 03_dev_gold.kpi_models.data_cube
GROUP BY country
ORDER BY country;

CREATE OR REPLACE VIEW 03_dev_gold.kpi_models.kpi_revenue_by_channel AS
SELECT
    order_channel AS channel,
    ROUND(SUM(total_amount),2) AS revenue
FROM 03_dev_gold.kpi_models.data_cube
GROUP BY order_channel
ORDER BY order_channel;


CREATE OR REPLACE VIEW 03_dev_gold.kpi_models.kpi_top_5_products AS
SELECT *
FROM 
(
    SELECT
        product_name,
        ROUND(SUM(total_amount),2) AS revenue,
        RANK() OVER (ORDER BY SUM(total_amount) DESC) AS rank
    FROM 03_dev_gold.kpi_models.data_cube 
    GROUP BY product_name
)
WHERE rank <= 5;


CREATE OR REPLACE VIEW 03_dev_gold.kpi_models.kpi_customer_acquistion AS
SELECT
    YEAR(registration_date) AS year,
    MONTH(registration_date) AS month,
    COUNT(DISTINCT customer_id) AS customers_acquired
FROM 03_dev_gold.kpi_models.data_cube
WHERE registration_date IS NOT NULL
GROUP BY YEAR(registration_date), MONTH(registration_date)
ORDER BY year, month;